In [ ]:
# Problem 4 (Intermediate-Advanced)
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('/content/adult_income_dataset.csv', na_values='?').drop(columns=['fnlwgt', 'edu_num'])
df = df.rename(columns={'eduaction': 'education'})

# Drop rows where the target variable 'income' is NaN
df = df.dropna(subset=['income'])

X = df.drop(columns=['income'])

# Convert target labels from strings to numerical values (0 and 1)
y = df['income'].map({'<=50K': 0, '>50K': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=34
)

numeric_features = ['age', 'capital_gain', 'capital_loss', 'hours_per_week']
nom_cat_features = ['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
ord_cat_features = ['education']

numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

nom_cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]
)

ord_cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=[['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th', 'HS-grad', 'Some-college', 'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters', 'Prof-school', 'Doctorate']]))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('nom', nom_cat_pipeline, nom_cat_features),
        ('ord', ord_cat_pipeline, ord_cat_features)
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2%}")
print(f"Precision Score: {precision_score(y_test, y_pred):.2%}")
print(f"Recall Score: {recall_score(y_test, y_pred):.2%}")
print(f"F1 Score: {f1_score(y_test, y_pred):.2%}")

joblib.dump(model, 'model.pkl')